# Streaming Event Producer

Generates fake order-activity events (page views, cart adds, purchases)
and writes them as JSON files into a volume, simulating a live event
feed. Run this repeatedly (or on a loop) while the streaming notebook
is running, to watch data flow through in near real time.

In [0]:
%pip install faker
dbutils.library.restartPython()

In [0]:
import json
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

from faker import Faker

fake = Faker()

EVENTS_DIR = Path("/Volumes/workspace/default/raw_data/streaming_events")
EVENTS_DIR.mkdir(parents=True, exist_ok=True)

EVENT_TYPES = ["page_view", "add_to_cart", "purchase"]
CATEGORIES = ["health_beauty", "electronics", "furniture_decor", "sports_leisure", "watches_gifts"]


def generate_event() -> dict:
    return {
        "event_id": str(uuid.uuid4()),
        "event_type": fake.random_element(EVENT_TYPES),
        "customer_id": fake.uuid4(),
        "product_category": fake.random_element(CATEGORIES),
        "amount": round(fake.random_number(digits=3) + fake.random.random(), 2),
        "event_timestamp": datetime.now(timezone.utc).isoformat(),
    }


def write_batch(batch_size: int = 20) -> None:
    # one file per batch - Auto Loader picks up new files as they land
    filename = f"events_{int(time.time() * 1000)}.json"
    events = [generate_event() for _ in range(batch_size)]

    with open(EVENTS_DIR / filename, "w") as f:
        for event in events:
            f.write(json.dumps(event) + "\n")

    print(f"wrote {batch_size} events to {filename}")

In [0]:
# run this cell repeatedly (or wrap in a loop with time.sleep) to simulate
# a continuous stream while the streaming notebook is running
write_batch()